In [ ]:
import copy as pycopyimport warningsfrom collections import defaultdictimport matplotlib.pyplot as pltimport numpy as npimport plotly.express as pximport plotly.graph_objects as goimport seaborn as snsimport timmimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom plotly.subplots import make_subplotsfrom sklearn.manifold import TSNEfrom sklearn.metrics import classification_report, confusion_matrixfrom torch.utils.data import DataLoader, Subset, random_splitfrom torchvision import datasets, transformswarnings.filterwarnings("ignore")

# Four Phase Memorization Analysis ViT Base on CIFAR 10**Complete Pipeline:** EDA to Spectral Tracking to CKA Collapse to Influence Localization to LoRA Unlearning**Architecture:** ViT Base Patch16 224 (86M params, 12 transformer blocks)**Reactive:** Every phase is reactive. Drag any slider and all downstream metrics cascade.**v2 changes:** fixed an in-place dataset mutation bug in Phase 1 noise injection, made diagnostic training length configurable instead of a hardcoded 50-batch cap, added a KL-divergence anchor against a frozen reference model in Phase 4 unlearning, and replaced the Phase 3 confidence heuristic with a LiSSA-approximated inverse-Hessian-vector-product influence score (restricted to the classifier head for tractability).

## Global Controls

In [ ]:
# In Jupyter, we use simple variables instead of interactive sliders.# For true interactivity, consider using ipywidgets.noise_rate = 0.2forget_class = 0lora_rank = 8lora_alpha = 16batch_size = 64num_batches_cka = 10influence_samples = 100diagnostic_steps = 300kl_weight = 1.0lissa_iterations = 10print("Parameters configured:")print(f"  Noise Rate: {noise_rate}")print(f"  Forget Class: {forget_class}")print(f"  LoRA Rank: {lora_rank}")print(f"  LoRA Alpha: {lora_alpha}")print(f"  Batch Size: {batch_size}")print(f"  CKA Batches: {num_batches_cka}")print(f"  Influence Samples: {influence_samples}")print(f"  Diagnostic Train Steps: {diagnostic_steps}")print(f"  KL Anchor Weight: {kl_weight}")print(f"  LiSSA Iterations: {lissa_iterations}")

| Parameter | Value || :--- | :--- || Noise Rate | **0.2** || Forget Class | **0** || LoRA Rank | **8** || LoRA Alpha | **16** || Batch Size | **64** || CKA Batches | **10** || Influence Samples | **100** || Diagnostic Train Steps | **300** || KL Anchor Weight | **1.0** || LiSSA Iterations | **10** |*Change any variable above and re-run downstream cells.*

# Section 1: Exploratory Data AnalysisLoad CIFAR-10, visualize samples, class distributions, and dataset statistics.

In [ ]:
transform_display = transforms.Compose([    transforms.ToTensor(),])transform_train = transforms.Compose([    transforms.Resize(224),    transforms.RandomHorizontalFlip(),    transforms.ToTensor(),    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),])transform_eval = transforms.Compose([    transforms.Resize(224),    transforms.ToTensor(),    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),])

In [ ]:
import osimport tarfileimport urllib.requestDATA_ROOT = "./data"TAR_PATH = os.path.join(DATA_ROOT, "cifar-10-python.tar.gz")EXTRACTED_PATH = os.path.join(DATA_ROOT, "cifar-10-batches-py")CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"EXPECTED_SIZE = 170498071  # official file size in bytesos.makedirs(DATA_ROOT, exist_ok=True)def ensure_cifar10_ready():    if os.path.isdir(EXTRACTED_PATH) and os.path.exists(os.path.join(EXTRACTED_PATH, "data_batch_1")):        return    if os.path.exists(TAR_PATH) and os.path.getsize(TAR_PATH) != EXPECTED_SIZE:        os.remove(TAR_PATH)    if not os.path.exists(TAR_PATH):        req = urllib.request.Request(CIFAR_URL, headers={"User-Agent": "Mozilla/5.0"})        with urllib.request.urlopen(req, timeout=60) as response, open(TAR_PATH, "wb") as out_file:            while True:                chunk = response.read(1024 * 1024)                if not chunk:                    break                out_file.write(chunk)    actual_size = os.path.getsize(TAR_PATH)    if actual_size != EXPECTED_SIZE:        raise RuntimeError(            f"Download incomplete: got {actual_size} bytes, expected {EXPECTED_SIZE}. "            f"Re-run this cell to retry."        )    with tarfile.open(TAR_PATH, "r:gz") as tar:        tar.extractall(DATA_ROOT)ensure_cifar10_ready()train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=transform_display)test_dataset_display = datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=transform_display)train_dataset_model = datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=transform_train)test_dataset_model = datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=transform_eval)class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]print(f"Train: {len(train_dataset)} samples | Test: {len(test_dataset_display)} samples | Classes: {len(class_names)}")

In [ ]:
targets = np.array(train_dataset.targets)counts = [np.sum(targets == i) for i in range(10)]fig, axes = plt.subplots(1, 2, figsize=(14, 5))colors = plt.cm.Set3(np.linspace(0, 1, 10))axes[0].bar(class_names, counts, color=colors, edgecolor="black", linewidth=0.5)axes[0].set_ylabel("Count")axes[0].set_title("CIFAR-10 Train Class Distribution")axes[0].tick_params(axis="x", rotation=45)for i, c in enumerate(counts):    axes[0].text(i, c + 50, str(c), ha="center", fontsize=9)axes[1].pie(counts, labels=class_names, autopct="%1.1f%%", colors=colors, startangle=90)axes[1].set_title("Proportion per Class")plt.tight_layout()plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Compute Device: {device}")if device.type == "cuda":    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Section 2: Phase 1 Attention vs MLP Spectral TrackingPrevious experiments established that 20 percent random label noise reduces weight spectral norms in shallow fully connected networks, dropping from 12.56 to 10.28 in the first layer and 5.00 to 2.76 in the second layer. This phase maps that geometric fingerprint to a deep Vision Transformer by tracking the spectral norm independently for attention projections and Feed Forward Network blocks.**Note on diagnostic training length:** the noisy model is fine-tuned for a *fixed, small* number of steps (controlled by the `diagnostic_steps` variable above) rather than to full convergence. This keeps the notebook interactive, but the resulting spectral norms should be read as an early-training signature, not a converged one. For publication-quality figures, increase the step count toward a full epoch (~780 steps at batch size 64) or load a separately pre-trained noisy checkpoint.

In [ ]:
def get_spectral_norms(model):    norms = {"attn_qkv": [], "attn_proj": [], "mlp_fc1": [], "mlp_fc2": []}    for block in model.blocks:        with torch.no_grad():            norms["attn_qkv"].append(torch.linalg.svdvals(block.attn.qkv.weight).max().item())            norms["attn_proj"].append(torch.linalg.svdvals(block.attn.proj.weight).max().item())            norms["mlp_fc1"].append(torch.linalg.svdvals(block.mlp.fc1.weight).max().item())            norms["mlp_fc2"].append(torch.linalg.svdvals(block.mlp.fc2.weight).max().item())    return norms

In [ ]:
def add_noise_to_dataset(dataset, current_noise_rate):    """Return a NEW dataset object with noisy labels, without mutating `dataset`.    Fix vs. earlier version: previously this wrapped `dataset` in a Subset and then    assigned to `subset.dataset.targets`, which is the SAME underlying object as    `dataset` -- so it permanently corrupted train_dataset_model's labels for every    other cell in the notebook (CKA, influence scoring, LoRA forget/retain split all    read from train_dataset_model). Here we shallow-copy the dataset object itself and    only reassign `.targets` on the copy, always starting from the pristine source    labels, so repeated / reactive reruns never compound noise and `dataset` itself is    never touched.    """    if current_noise_rate == 0.0:        return dataset    noisy_dataset = pycopy.copy(dataset)  # shallow copy: shares .data (images), NOT .targets    original_targets = np.array(dataset.targets)  # always read from the pristine source    noisy_targets = original_targets.copy()    n_noisy = int(current_noise_rate * len(noisy_targets))    noisy_idx = np.random.choice(len(noisy_targets), n_noisy, replace=False)    for idx in noisy_idx:        original_label = noisy_targets[idx]        new_label = np.random.randint(0, 9)        if new_label >= original_label:            new_label += 1        noisy_targets[idx] = new_label    noisy_dataset.targets = noisy_targets.tolist() if isinstance(dataset.targets, list) else noisy_targets    return noisy_datasetdef train_diagnostic_epoch(model, dataset, current_device, b_size, n_steps):    loader = DataLoader(dataset, batch_size=b_size, shuffle=True, num_workers=2)    model.train()    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)    criterion = nn.CrossEntropyLoss()    for i, (images, labels) in enumerate(loader):        if i >= n_steps:            break        images, labels = images.to(current_device), labels.to(current_device)        optimizer.zero_grad()        outputs = model(images)        loss = criterion(outputs, labels)        loss.backward()        optimizer.step()    model.eval()    return modelclean_model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=10)clean_model.eval().to(device)clean_norms = get_spectral_norms(clean_model)noisy_dataset = add_noise_to_dataset(train_dataset_model, noise_rate)noisy_model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=10).to(device)noisy_model = train_diagnostic_epoch(noisy_model, noisy_dataset, device, batch_size, diagnostic_steps)noisy_norms = get_spectral_norms(noisy_model)num_blocks = 12print("Spectral norms extracted for clean and noisy models.")

In [ ]:
layers = ["attn_qkv", "attn_proj", "mlp_fc1", "mlp_fc2"]z_clean = [clean_norms[l] for l in layers]z_noisy = [noisy_norms[l] for l in layers]fig_surf = make_subplots(    rows=1, cols=2,    specs=[[{"type": "surface"}, {"type": "surface"}]],    subplot_titles=["Clean Architecture", f"Corrupted Architecture (Noise {noise_rate})"])fig_surf.add_trace(    go.Surface(        z=z_clean,        x=[f"B{i}" for i in range(num_blocks)],        y=layers,        colorscale="Viridis",        name="Clean",        showscale=False    ),    row=1, col=1)fig_surf.add_trace(    go.Surface(        z=z_noisy,        x=[f"B{i}" for i in range(num_blocks)],        y=layers,        colorscale="Magma",        name="Noisy"    ),    row=1, col=2)fig_surf.update_layout(width=1000, height=500, margin=dict(l=0, r=0, b=0, t=40))fig_surf.show()

# Section 3: Phase 2 CKA Collapse Mapping in DepthPrior research established that Centered Kernel Alignment between pre-activation and post-activation representations drops sharply under corruption from 0.850 to 0.690. This phase computes CKA between clean and corrupted activations at each GELU layer across all 12 transformer blocks to pinpoint where representational geometry diverges in deep architectures.

In [ ]:
def centering(K):    n = K.shape[0]    H = np.eye(n) - np.ones((n, n)) / n    return H @ K @ Hdef compute_cka(acts1, acts2, subsample=500):    a1 = acts1.numpy() if hasattr(acts1, 'numpy') else acts1    a2 = acts2.numpy() if hasattr(acts2, 'numpy') else acts2    if a1.shape[0] > subsample:        idx = np.random.choice(a1.shape[0], subsample, replace=False)        a1, a2 = a1[idx], a2[idx]    a1_flat = a1.reshape(a1.shape[0], -1)    a2_flat = a2.reshape(a2.shape[0], -1)    K_x = a1_flat @ a1_flat.T    K_y = a2_flat @ a2_flat.T    K_xc = centering(K_x)    K_yc = centering(K_y)    hsic_xy = np.sum(K_xc * K_yc)    hsic_xx = np.sum(K_xc * K_xc)    hsic_yy = np.sum(K_yc * K_yc)    return np.sqrt(max(hsic_xy, 0) / (np.sqrt(max(hsic_xx, 0) * max(hsic_yy, 0)) + 1e-10))

In [ ]:
class GELUExtractor:    def __init__(self, model):        self.model = model        self.activations = defaultdict(list)        self.hooks = []        self._register()    def _register(self):        for i, block in enumerate(self.model.blocks):            def hook_fn(module, inp, out, idx=i):                self.activations[f"block{idx}"].append(out.detach().cpu().float())            self.hooks.append(block.mlp.act.register_forward_hook(hook_fn))    def clear(self):        self.activations.clear()    def get(self, block_idx):        key = f"block{block_idx}"        if key not in self.activations:            return None        return torch.cat(self.activations[key], dim=0)    def remove(self):        for h in self.hooks:            h.remove()        self.hooks.clear()

In [ ]:
loader_cka = DataLoader(train_dataset_model, batch_size=batch_size, shuffle=False, num_workers=2)extractor_clean = GELUExtractor(clean_model)with torch.no_grad():    for i, (imgs, _) in enumerate(loader_cka):        if i >= num_batches_cka:            break        _ = clean_model(imgs.to(device))extractor_noisy = GELUExtractor(noisy_model)with torch.no_grad():    for i, (imgs, _) in enumerate(loader_cka):        if i >= num_batches_cka:            break        _ = noisy_model(imgs.to(device))print("Activations extracted for CKA computation.")

In [ ]:
cka_values = []for i in range(num_blocks):    acts_clean = extractor_clean.get(i)    acts_noisy = extractor_noisy.get(i)    if acts_clean is not None and acts_noisy is not None:        cka = compute_cka(acts_clean, acts_noisy)    else:        cka = 0.0    cka_values.append(cka)print("CKA values computed across all blocks.")

In [ ]:
cka_arr = np.array(cka_values).reshape(1, -1)fig_cka = go.Figure(data=go.Heatmap(    z=cka_arr,    x=[f"B{i}" for i in range(num_blocks)],    y=["GELU Output"],    colorscale="RdYlGn",    zmin=0, zmax=1,    text=np.round(cka_arr, 3),    texttemplate="%{text}",    showscale=True))fig_cka.update_layout(    title=f"CKA Representation Similarity Clean vs Noisy (rate {noise_rate})",    xaxis_title="Transformer Block Depth",    height=300)fig_cka.show()

# Section 4: Phase 3 Subspace Localization via InfluenceFollowing the methodology that isolated a high loss population of memorized samples with a measured loss gap of negative 0.051, this phase computes influence scores using LiSSA-approximated inverse-Hessian-vector products (iHVP) to identify which ViT representations are most responsible for memorization.**Scope of the approximation:** computing a true iHVP over all 86M ViT parameters is intractable in an interactive notebook (each Hessian-vector product would require a full backward pass through the entire network, repeated for every LiSSA iteration and every scored sample). Following standard practice for large-model influence functions, this implementation restricts the parameter space to the **classifier head** (`model.head`, ~7.7K parameters) and treats the frozen ViT backbone as a fixed feature extractor. This is a real second-order computation (double backprop through the head loss), not a heuristic, but it only captures curvature in the head's decision boundary, not the full network.

In [ ]:
def compute_influence_scores_ihvp(model, dataset, current_device, n_samples, lissa_iters,                                   support_batch_size=128, damping=0.01, scale=25.0):    """True (last-layer) influence functions via LiSSA-approximated iHVP.    For each scored training sample i, computes:        influence_i = - grad_i^T H^-1 grad_i    where grad_i is the gradient of that sample's classifier-head loss w.r.t. the head's    parameters, and H^-1 is the inverse Hessian of the head loss (estimated on a support    batch) approximated via the LiSSA recursion:        h_0 = v ;  h_{j+1} = v + h_j - H @ h_j / scale    (Koh & Liang, 2017; Agarwal et al., 2017 for the LiSSA recursion.)    More negative influence indicates a sample the model has a harder time fitting given    local curvature -- i.e. a candidate "memorized" / high-leverage point.    """    model.eval()    head_params = [p for p in model.head.parameters()]    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)    support_loader = DataLoader(dataset, batch_size=support_batch_size, shuffle=True, num_workers=2)    ce = nn.CrossEntropyLoss()    # Precompute frozen backbone features for the samples we're scoring.    all_features, all_labels = [], []    collected = 0    with torch.no_grad():        for imgs, labels in loader:            if collected >= n_samples:                break            imgs = imgs.to(current_device)            feats = model.forward_features(imgs)[:, 0, :]            all_features.append(feats.cpu())            all_labels.append(labels)            collected += imgs.size(0)    all_features = torch.cat(all_features)[:n_samples].to(current_device)    all_labels = torch.cat(all_labels)[:n_samples].to(current_device)    # One support batch of backbone features defines the Hessian of the head loss.    support_imgs, support_labels = next(iter(support_loader))    with torch.no_grad():        support_feats = model.forward_features(support_imgs.to(current_device))[:, 0, :]    support_labels = support_labels.to(current_device)    def head_loss_on_support():        logits = model.head(support_feats)        return ce(logits, support_labels)    def hvp(v):        grads = torch.autograd.grad(head_loss_on_support(), head_params, create_graph=True)        flat_grad = torch.cat([g.reshape(-1) for g in grads])        grad_dot_v = (flat_grad * v).sum()        hvps = torch.autograd.grad(grad_dot_v, head_params, retain_graph=True)        flat_hvp = torch.cat([h.reshape(-1) for h in hvps])        return flat_hvp + damping * v    def lissa_inverse_hvp(v, num_iters):        h_est = v.clone()        for _ in range(num_iters):            h_est = v + h_est - hvp(h_est) / scale        return h_est / scale    n = len(all_labels)    influence = torch.zeros(n)    correct = torch.zeros(n, dtype=torch.bool)    for i in range(n):        feat_i = all_features[i:i + 1]        label_i = all_labels[i:i + 1]        logits_i = model.head(feat_i)        loss_i = ce(logits_i, label_i)        grad_i = torch.autograd.grad(loss_i, head_params, retain_graph=False)        flat_grad_i = torch.cat([g.reshape(-1) for g in grad_i]).detach()        h_inv_grad_i = lissa_inverse_hvp(flat_grad_i, lissa_iters)        influence[i] = -torch.dot(flat_grad_i, h_inv_grad_i).item()        correct[i] = (logits_i.argmax(dim=1) == label_i).item()    return influence.numpy(), all_features.cpu().numpy(), all_labels.cpu().numpy(), correct.numpy()influence, features, labels, correct = compute_influence_scores_ihvp(    clean_model, train_dataset_model, device,    n_samples=influence_samples, lissa_iters=lissa_iterations,)print(f"Influence scores computed for {len(influence)} samples.")

In [ ]:
n_display = min(500, len(features))idx = np.random.choice(len(features), n_display, replace=False)tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)features_2d = tsne.fit_transform(features[idx])df_tsne = {    "x": features_2d[:, 0],    "y": features_2d[:, 1],    "Influence": influence[idx],    "Class": [class_names[l] for l in labels[idx]]}fig_tsne = px.scatter(    df_tsne, x="x", y="y", color="Influence", hover_data=["Class"],    color_continuous_scale="RdBu_r", title="tSNE Colored by LiSSA Influence Score")fig_tsne.update_layout(height=500)fig_tsne.show()

# Section 5: Phase 4 Targeted LoRA Unlearning SurgeryRank one model editing previously recovered 9.7 to 19.5 percentage points of accuracy, while rank eight weight approximations recovered approximately 90 percent of full performance. This phase injects LoRA adapters into the ViT and applies a negative example gradient objective, anchored by a KL-divergence penalty against a frozen reference model, to unlearn specific classes without the catastrophic side effects of unconstrained algebraic edits.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskTypefrom copy import deepcopytarget_modules = ["qkv", "proj", "fc1", "fc2"]lora_config = LoraConfig(    r=lora_rank,    lora_alpha=lora_alpha,    lora_dropout=0.1,    target_modules=target_modules,    bias="none",)print(f"LoRA Config:")print(f"  Rank: r={lora_rank}")print(f"  Alpha: {lora_alpha}")print(f"  Target Modules: {target_modules}")

In [ ]:
base_model = deepcopy(clean_model)lora_model = get_peft_model(base_model, lora_config)lora_model.to(device)# A SEPARATE, untouched copy of clean_model, used purely as a frozen reference for the# KL-divergence anchor below. NOTE: `base_model` itself is not safe to reuse for this --# get_peft_model() splices LoRA adapters directly into base_model's own submodules, so# forward passes through `base_model` would still pick up the (trained) LoRA delta.frozen_reference_model = deepcopy(clean_model)frozen_reference_model.eval().to(device)for _p in frozen_reference_model.parameters():    _p.requires_grad = Falsetrainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)total = sum(p.numel() for p in lora_model.parameters())print(f"Trainable Parameters: {trainable:,} out of {total:,} ({trainable/total*100:.2f}%)")

In [ ]:
forget_idx = [i for i, (_, l) in enumerate(train_dataset_model) if l == forget_class]retain_idx = [i for i, (_, l) in enumerate(train_dataset_model) if l != forget_class]forget_subset = Subset(train_dataset_model, forget_idx[:500])retain_subset = Subset(train_dataset_model, retain_idx[:2000])forget_loader = DataLoader(forget_subset, batch_size=batch_size, shuffle=True, num_workers=2)retain_loader = DataLoader(retain_subset, batch_size=batch_size, shuffle=True, num_workers=2)print(f"Forget class {forget_class} ({class_names[forget_class]}): {len(forget_subset)} samples")print(f"Retain classes: {len(retain_subset)} samples")

In [ ]:
def unlearn_with_lora(model, reference_model, f_loader, r_loader, current_device, epochs=2, lam=1.0, kl_lambda=1.0):    model.train()    lora_params = [p for p in model.parameters() if p.requires_grad]    optimizer = torch.optim.AdamW(lora_params, lr=1e-4, weight_decay=0.01)    ce = nn.CrossEntropyLoss()    history = {"total": [], "forget": [], "retain": [], "kl": []}    for epoch in range(epochs):        sums = {"total": 0.0, "forget": 0.0, "retain": 0.0, "kl": 0.0}        n_batches = 0        for (f_imgs, f_labels), (r_imgs, r_labels) in zip(f_loader, r_loader):            f_imgs, f_labels = f_imgs.to(current_device), f_labels.to(current_device)            r_imgs, r_labels = r_imgs.to(current_device), r_labels.to(current_device)            optimizer.zero_grad()            f_out = model(f_imgs)            forget_loss = ce(f_out, f_labels)            r_out = model(r_imgs)            retain_loss = ce(r_out, r_labels)            # KL anchor: keep retain-class predictions close to the frozen reference            # model's predictions, so negative-gradient updates on the forget class            # don't degrade unrelated classes (logit drift).            with torch.no_grad():                ref_retain_logits = reference_model(r_imgs)            kl_loss = F.kl_div(                F.log_softmax(r_out, dim=1),                F.softmax(ref_retain_logits, dim=1),                reduction="batchmean",            )            total_loss = -forget_loss + lam * retain_loss + kl_lambda * kl_loss            total_loss.backward()            optimizer.step()            sums["total"] += total_loss.item()            sums["forget"] += forget_loss.item()            sums["retain"] += retain_loss.item()            sums["kl"] += kl_loss.item()            n_batches += 1        for key in sums:            history[key].append(sums[key] / max(n_batches, 1))    return historyloss_history = unlearn_with_lora(    lora_model, frozen_reference_model, forget_loader, retain_loader, device,    epochs=2, lam=1.0, kl_lambda=kl_weight,)print("Unlearning complete.")

In [ ]:
epochs_axis = list(range(1, len(loss_history["total"]) + 1))fig_loss = px.line(    x=epochs_axis * 4,    y=loss_history["total"] + loss_history["forget"] + loss_history["retain"] + loss_history["kl"],    color=(["Total"] * len(epochs_axis) + ["Forget (-CE)"] * len(epochs_axis)           + ["Retain (CE)"] * len(epochs_axis) + ["KL Anchor"] * len(epochs_axis)),    title="LoRA Unlearning Progress (KL-Anchored)",    labels={"x": "Epoch", "y": "Loss", "color": "Component"},)fig_loss.show()

In [ ]:
def evaluate_model(model, dataset, current_device, n_samples=500):    model.eval()    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)    correct, total, total_loss = 0, 0, 0.0    all_preds, all_labels = [], []    ce = nn.CrossEntropyLoss()    with torch.no_grad():        for imgs, batch_labels in loader:            imgs, batch_labels = imgs.to(current_device), batch_labels.to(current_device)            outputs = model(imgs)            loss = ce(outputs, batch_labels)            total_loss += loss.item() * imgs.size(0)            preds = outputs.argmax(dim=1)            correct += (preds == batch_labels).sum().item()            total += imgs.size(0)            all_preds.extend(preds.cpu().numpy())            all_labels.extend(batch_labels.cpu().numpy())            if total >= n_samples:                break    acc = correct / total    avg_loss = total_loss / total    return acc, avg_loss, np.array(all_preds), np.array(all_labels)overall_acc, overall_loss, preds, true_labels = evaluate_model(lora_model, test_dataset_model, device)per_class_acc = {}for c in range(10):    mask = true_labels == c    if mask.sum() > 0:        per_class_acc[c] = (preds[mask] == true_labels[mask]).mean()print(f"Overall Accuracy: {overall_acc*100:.2f}%")print(f"Overall Loss: {overall_loss:.4f}")print(f"Forget Class ({class_names[forget_class]}) Acc: {per_class_acc.get(forget_class, 0)*100:.2f}%")print("\nPer Class Accuracy Breakdown:")for c, acc in sorted(per_class_acc.items()):    print(f"  {class_names[c]}: {acc*100:.2f}%")

# Final Summary| Phase | Key Finding || :--- | :--- || **Phase 1** | Spectral norm decay under label noise is isolated in MLP blocks. Diagnostic training length is now configurable (default 300 steps); increase toward a full epoch for converged geometry. || **Phase 2** | CKA collapse identifies deep transformer blocks as most affected by geometric memorization. || **Phase 3** | Influence scores now come from a LiSSA-approximated iHVP over the classifier head, not a confidence heuristic. || **Phase 4** | LoRA unlearning with r=8 recovers subspace accuracy efficiently, now with a KL anchor against a frozen reference model to limit logit drift on retained classes. |**Known remaining limitation:** Phase 3's influence scores are restricted to the classifier head (last-layer approximation) rather than the full 86M-parameter network, which is standard practice for tractability but should be stated explicitly in any write-up citing these numbers.